## 07_annotate.ipynb — Phase C: Annotation set preparation

Draws a stratified sample of 2,500 posts for manual harm labelling. Sampling is
weighted toward the harm-concentrating clusters (Conflict-Exposed, High-Intensity)
so the fixed labelling budget captures the rare severe cases efficiently — an
expected ~256 severe cases vs ~125 from random sampling of the same size.

### Generate the annotation set

Allocates the 2,500 posts across clusters (per the stratification math), then within
each cluster picks 70% by harm-signal strength (so real harm cases surface, not empty
chatter) and 30% at random (for coverage and clean negatives). The final set is
shuffled so labelling isn't biased by cluster order, and written with blank
`parasocial_risk` / `bullying` / `notes` columns to `to_annotate.csv`.

In [ ]:
import os, pandas as pd, numpy as np
os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")

# load Phase B clusters + scored posts
cl = pd.read_parquet("data/processed/user_clusters.parquet")
df = pd.read_parquet("data/processed/posts_scored.parquet")

# 2,500 stratified allocation (from the sampling math)
alloc = {"Financially Intensive Collector":317, "Casual Participant":205,
         "Devoted":289, "Conflict-Exposed":651, "High-Intensity Multi-Domain":568,
         "Peripheral":177, "Broadly Engaged":289}

# tag each post with its user's cluster type
u2type = cl.set_index("uid").type
df["type"] = df.uid.map(u2type)

# harm-signal columns for signal-weighted sampling
harmcols = [c for c in df.columns if c.startswith(("victim_","financial_","parasocial_"))]

rng = np.random.default_rng(0)
picks = []
for ctype, n in alloc.items():
    pool = df[df.type == ctype].copy()
    if len(pool) == 0:
        print(f"WARNING: no posts for {ctype}"); continue
    pool["signal"] = pool[harmcols].sum(1)
    # 70% signal-weighted (so annotators see real harm cases), 30% random (coverage)
    n_sig = int(n * 0.7)
    hi = pool.nlargest(n_sig * 3, "signal").sample(min(n_sig, len(pool)), random_state=0)
    rest = pool.drop(hi.index).sample(min(n - len(hi), len(pool) - len(hi)), random_state=0)
    picks.append(pd.concat([hi, rest]))

ann = pd.concat(picks).sample(frac=1, random_state=0).reset_index(drop=True)  # shuffle
ann = ann[["uid", "subreddit", "type", "text"]].copy()

# blank columns you'll fill during annotation
ann["parasocial_risk"] = ""    # none / moderate / acute
ann["bullying"] = ""           # none / involved / targeted
ann["financial_risk"] = ""     # none / moderate / acute
ann["notes"] = ""

ann.to_csv("data/processed/to_annotate.csv", index=False)
print("wrote to_annotate.csv —", len(ann), "posts")
print(ann.type.value_counts())

/tmp/ipykernel_194826/1635831751.py:26: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  pool["signal"] = pool[harmcols].sum(1)
/tmp/ipykernel_194826/1635831751.py:26: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  pool["signal"] = pool[harmcols].sum(1)
/tmp/ipykernel_194826/1635831751.py:26: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  pool["signal"] = pool[harmcols].sum(1)
/tmp/ipykernel_194826/1635831751.py:26: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  pool["signal"] = pool[harmcols].sum(1)
/tmp/ipykernel_194826/1635831751.py:26: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  pool["signal"] = pool[harmcols].sum(1)


wrote to_annotate.csv — 2496 posts
type
Conflict-Exposed                   651
High-Intensity Multi-Domain        568
Financially Intensive Collector    317
Broadly Engaged                    289
Devoted                            289
Casual Participant                 205
Peripheral                         177
Name: count, dtype: int64


/tmp/ipykernel_194826/1635831751.py:26: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  pool["signal"] = pool[harmcols].sum(1)
/tmp/ipykernel_194826/1635831751.py:26: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  pool["signal"] = pool[harmcols].sum(1)


### Check the sample

Confirms the per-cluster counts match the allocation and that harm-cluster posts
look like genuine conflict/harm content. If any cluster warns "no posts" or comes up
short, its allocation exceeded available posts and needs redistributing.

In [2]:
ann = pd.read_csv("data/processed/to_annotate.csv")
print("total:", len(ann))
print(ann.type.value_counts())
# peek at a few Conflict-Exposed posts to confirm they look like real conflict
print(ann[ann.type=="Conflict-Exposed"].text.head(5).tolist())

total: 2496
type
Conflict-Exposed                   651
High-Intensity Multi-Domain        568
Financially Intensive Collector    317
Broadly Engaged                    289
Devoted                            289
Casual Participant                 205
Peripheral                         177
Name: count, dtype: int64
['you give me stan twt vibes 😭.', "personally i think it's the fancams. fancams everywhere. fancams on non-kpop related stuff, fancams on posts where it's wildly inappropriate. it's embarrassing and won't help your group at all", 'my pace, a few days after its release. i kept it in my playlist because it made me so happy, then became a stay. simple', 'i heard fancams included as well.', 'ah, i understand now. sorry.']
